# Superstore Sales - 探索性分析 (EDA)

先看看数据的整体情况，搞清楚几个问题：
1. 销售额有没有趋势？季节性怎么样？
2. 不同品类和区域的差异大不大？
3. 序列是否平稳？需要几阶差分？

这些结论直接影响后面建模的选择。

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from src.data_loader import load_raw_data, get_monthly_sales, get_category_monthly

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

%matplotlib inline

## 1. 数据概览

先把数据加载进来，看看基本结构和字段类型。

In [ ]:
df = load_raw_data('../data/raw/train.csv')
print(f'数据量: {len(df)} rows')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# 看看日期范围和分类字段
print(f"Order Date range: {df['Order Date'].min()} ~ {df['Order Date'].max()}")
print(f"\nCategories: {df['Category'].unique()}")
print(f"Sub-Categories: {df['Sub-Category'].nunique()} types")
print(f"Regions: {df['Region'].unique()}")
print(f"Segments: {df['Segment'].unique()}")

4年数据，3个品类，4个区域。数据量不算大，聚合到月度之后只有48个点，这对后面LSTM之类的深度模型不太友好。

## 2. 月度销售趋势

聚合到月粒度看整体走势。这是后面预测的目标变量。

In [ ]:
monthly = get_monthly_sales(df)
print(f'Monthly series: {len(monthly)} data points')
monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly.values, marker='o', markersize=3, color='steelblue')
ax.set_title('Monthly Total Sales')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

两个观察：
1. **趋势向上** — 2018年整体比2015年高不少
2. **年末spike** — 每年11-12月都有明显高峰，应该是Black Friday / Christmas的效应

这种pattern对SARIMA和Prophet来说应该比较好处理。

## 3. 按品类拆分

看看三个品类的趋势是不是一致的，如果差异很大可能需要分别建模。

In [ ]:
cat_monthly = get_category_monthly(df)

fig, ax = plt.subplots(figsize=(14, 5))
for cat in cat_monthly['category'].unique():
    mask = cat_monthly['category'] == cat
    subset = cat_monthly[mask]
    ax.plot(subset['date'], subset['sales'], label=cat, marker='o', markersize=2)
ax.legend()
ax.set_title('Monthly Sales by Category')
ax.set_ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/category_trend.png', dpi=150, bbox_inches='tight')
plt.show()

Technology的波动最大，偶尔会有单月暴涨（可能是大额订单）。Office Supplies比较稳定，Furniture居中。

先用总量做预测，后面如果效果不好可以考虑按品类分别建模再加总。

## 4. 区域和品类分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=True)
region_sales.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Total Sales by Region')
axes[0].set_xlabel('Sales ($)')

cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=True)
cat_sales.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Total Sales by Category')
axes[1].set_xlabel('Sales ($)')

plt.tight_layout()
plt.savefig('../results/figures/region_category_bar.png', dpi=150, bbox_inches='tight')
plt.show()

West和East占了大头，South最少。品类方面Technology总额最高但订单量不一定最多，可能是客单价高。

## 5. STL季节性分解

用STL（Seasonal and Trend decomposition using Loess）把信号拆成trend + seasonal + residual三部分。

In [ ]:
stl = STL(monthly, period=12)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.savefig('../results/figures/stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

分解结果：
- **Trend**: 稳步上升，2017年之后加速
- **Seasonal**: 很规律，振幅大概在±15k左右，说明季节性效应不小
- **Residual**: 没有特别明显的pattern，个别月份偏大但整体还好

季节性这么强的话，Seasonal Naive可能就已经不错了。

## 6. ACF / PACF

自相关图帮我们确认ARIMA的参数。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(monthly, lags=24, ax=axes[0])
axes[0].set_title('ACF')

plot_pacf(monthly, lags=24, ax=axes[1], method='ywm')
axes[1].set_title('PACF')

plt.tight_layout()
plt.savefig('../results/figures/acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

ACF缓慢衰减 → 有趋势成分需要差分。lag 12附近有个明显的bump → 年度季节性。PACF在lag 1处显著 → AR(1)可能就够了。

初步判断SARIMA的参数大概是 (1,1,0)(0,1,1,12) 这个量级，具体让auto_arima去搜。

## 7. 平稳性检验 (ADF Test)

正式用ADF test验证一下我们的判断。

In [ ]:
def run_adf(series, name=''):
    result = adfuller(series, autolag='AIC')
    print(f'ADF Test: {name}')
    print(f'  Statistic: {result[0]:.4f}')
    print(f'  p-value:   {result[1]:.4f}')
    print(f'  Stationary: {"Yes" if result[1] < 0.05 else "No (p > 0.05)"}')
    print()

run_adf(monthly, 'Original')
run_adf(monthly.diff().dropna(), '1st Difference')
run_adf(monthly.diff(12).dropna(), 'Seasonal Diff (lag=12)')

原始序列不平稳，一阶差分之后平稳了。ARIMA里 d=1 应该够用。

## 8. 月度分布

最后用boxplot看看每个月的销售分布，确认季节性pattern。

In [ ]:
monthly_df = pd.DataFrame({'sales': monthly})
monthly_df['month'] = monthly_df.index.month

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=monthly_df, x='month', y='sales', ax=ax, color='lightblue')
ax.set_title('Sales Distribution by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('../results/figures/monthly_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

9月和11-12月最高，1-2月最低。跟零售行业的经验吻合：开学季 + 年末促销。

---

### 小结

| 发现 | 对建模的影响 |
|------|-------------|
| 上升趋势 | 需要差分或者在模型中显式建模trend |
| 年度季节性 (period=12) | SARIMA用m=12，Prophet自动处理 |
| 数据量小 (48点) | LSTM可能学不好，统计模型更合适 |
| Technology波动大 | 聚合后被平滑了，暂时用总量建模 |

接下来去 `02_baseline_models.ipynb` 跑SARIMA和Prophet。